In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("StockTweetProcessing") \
    .getOrCreate()

spark

26/05/15 13:47:19 WARN Utils: Your hostname, Chakuunaa resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/15 13:47:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/15 13:47:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [8]:
import os

os.getcwd()

'/home/chakuunaa/BD_ADA_CA2sem2/notebooks'

In [10]:
tweets_spark = spark.read.csv(
    "file:///home/chakuunaa/BD_ADA_CA2sem2/data/stocktweet.csv",
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"'
)

tweets_spark.show(5, truncate=False)

+------+----------+------+-------------------------------------------------------------------------------------------------------------------------------------------+
|id    |date      |ticker|tweet                                                                                                                                      |
+------+----------+------+-------------------------------------------------------------------------------------------------------------------------------------------+
|100001|01/01/2020|AMZN  |$AMZN Dow futures up by 100 points already 🥳                                                                                              |
|100002|01/01/2020|TSLA  |$TSLA Daddy's drinkin' eArly tonight! Here's to a PT of ohhhhh $1000 in 2020! 🍻                                                           |
|100003|01/01/2020|AAPL  |$AAPL We’ll been riding since last December from $172.12 what to do. Decisions decisions hmm 🤔. I have 20 mins to decide. Any suggestions?|
|

In [11]:
tweets_spark.printSchema()

root
 |-- id: integer (nullable = true)
 |-- date: string (nullable = true)
 |-- ticker: string (nullable = true)
 |-- tweet: string (nullable = true)



In [12]:
tweets_spark.count()

10000

In [13]:
aapl_spark = spark.read.csv(
    "file:///home/chakuunaa/BD_ADA_CA2sem2/data/stockprice/AAPL.csv",
    header=True,
    inferSchema=True
)

aapl_spark.show(5)

+----------+-----------------+-----------------+-----------------+-----------------+-----------------+---------+
|      Date|             Open|             High|              Low|            Close|        Adj Close|   Volume|
+----------+-----------------+-----------------+-----------------+-----------------+-----------------+---------+
|2019-12-31|72.48249816894531|73.41999816894531|72.37999725341797| 73.4124984741211|71.52082061767578|100805600|
|2020-01-02|74.05999755859375| 75.1500015258789|73.79750061035156| 75.0875015258789|73.15264892578125|135480400|
|2020-01-03| 74.2874984741211| 75.1449966430664|           74.125|74.35749816894531|72.44145965576172|146322800|
|2020-01-06|73.44750213623047|74.98999786376953|          73.1875|74.94999694824219| 73.0186767578125|118387200|
|2020-01-07|74.95999908447266| 75.2249984741211|74.37000274658203|74.59750366210938|72.67527770996094|108872000|
+----------+-----------------+-----------------+-----------------+-----------------+------------

In [14]:
from pyspark.sql.functions import (
    lower,
    regexp_replace,
    col,
    trim
)

In [15]:
cleaned_tweets = tweets_spark.withColumn(
    "clean_tweet",
    lower(col("tweet"))
)

cleaned_tweets = cleaned_tweets.withColumn(
    "clean_tweet",
    regexp_replace(col("clean_tweet"), r"http\\S+", "")
)

cleaned_tweets = cleaned_tweets.withColumn(
    "clean_tweet",
    regexp_replace(col("clean_tweet"), r"[^a-zA-Z0-9\\s]", "")
)

cleaned_tweets = cleaned_tweets.withColumn(
    "clean_tweet",
    trim(col("clean_tweet"))
)

cleaned_tweets.select(
    "tweet",
    "clean_tweet"
).show(5, truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------+
|tweet                                                                                                                                      |clean_tweet                                                                                               |
+-------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------+
|$AMZN Dow futures up by 100 points already 🥳                                                                                              |amzndowfuturesupby100pointsalready                                                                        |
|$TSL

In [16]:
cleaned_tweets.filter(
    col("clean_tweet").isNull()
).count()

0

In [17]:
tweet_counts = cleaned_tweets.groupBy(
    "date",
    "ticker"
).count()

tweet_counts.show(10)

+----------+------+-----+
|      date|ticker|count|
+----------+------+-----+
|24/01/2020|  TSLA|    7|
|29/01/2020|   CVX|    1|
|03/02/2020|  TSLA|   15|
|10/03/2020|  MSFT|    1|
|24/03/2020|  AMZN|    5|
|25/03/2020|   NKE|    1|
|31/03/2020|   WMT|    1|
|08/04/2020|  AAPL|    3|
|01/07/2020|  TSLA|    5|
|31/07/2020|   CCL|    2|
+----------+------+-----+
only showing top 10 rows



In [18]:
company_activity = cleaned_tweets.groupBy(
    "ticker"
).count().orderBy(
    col("count").desc()
)

company_activity.show()

+------+-----+
|ticker|count|
+------+-----+
|  TSLA| 4341|
|  AAPL| 1721|
|    BA|  919|
|   DIS|  432|
|  AMZN|  407|
|  MSFT|  271|
|   CCL|  264|
|  BABA|  239|
|    FB|  204|
|  NFLX|  195|
|   PFE|  186|
|  NVDA|  119|
|   WMT|  118|
|   BAC|   65|
|  ABNB|   60|
|   XOM|   46|
|     V|   45|
|  SBUX|   45|
|    HD|   39|
|  PYPL|   38|
+------+-----+
only showing top 20 rows



In [19]:
tweet_counts.write.mode("overwrite").csv(
    "file:///home/chakuunaa/BD_ADA_CA2sem2/data/processed_tweet_counts",
    header=True
)

In [22]:
sc = spark.sparkContext

In [23]:
# sc master - running locally
sc.master

'local[*]'

In [24]:
# Import regex module
import re
from operator import add

In [27]:
# Read input file from hadoop directory
# Copy a file from the local folder to hadoop folder
file_in = sc.textFile('file:///home/chakuunaa/BD_ADA_CA2sem2/data/stocktweet.csv')

In [28]:
# Count lines
print('number of lines in file: %s' % file_in.count())

[Stage 22:>                                                         (0 + 2) / 2]

number of lines in file: 11581


In [29]:
# Get words from the input file
words = file_in.flatMap(lambda line: re.split('\\W+', line.lower().strip()))

In [30]:
# Keep words of more than 3 charaters
words = words.filter(lambda x: len(x) > 3)

In [31]:
# Set count 1 per word
words = words.map(lambda w: (w,1))

In [32]:
# Reduce phase - sum count all the words
words = words.reduceByKey(add)

In [33]:
# Create tuple (count, word) and sort in desceding order
words = words.map(lambda x: (x[1],x[0])).sortByKey(False)

In [34]:
# Take the top 20 words by frequency
words.take(10)

[(10030, '2020'),
 (8769, 'tsla'),
 (3477, 'aapl'),
 (2321, 'this'),
 (1171, 'will'),
 (915, 'that'),
 (823, 'amzn'),
 (681, 'just'),
 (673, 'today'),
 (641, 'going')]